# SSL DAPT + LoRA Training on Qwen3 Models

This notebook demonstrates **Domain-Adaptive Pre-Training (DAPT)** with LoRA on Qwen3-4B or Qwen3-8B base models.

## What is DAPT?

**Domain-Adaptive Pre-Training** continues pre-training a language model on domain-specific corpus to inject new knowledge while preserving existing capabilities.

## DAPT Best Practices

1. **Use BASE models** (not instruct versions) for better knowledge absorption
2. **Lower learning rate** (1e-5 to 5e-5) to prevent catastrophic forgetting
3. **Longer warmup** (10%) for stable training
4. **Full precision** recommended (quantization optional for memory constraints)
5. **Higher LoRA rank** for more capacity to learn new knowledge
6. **Sequence packing** for efficient GPU utilization

## 1. Install Dependencies

In [ ]:
!pip install -q torch transformers datasets accelerate peft bitsandbytes wandb tqdm

## 2. Import Libraries

In [ ]:
import os
import json
import random
import logging
from pathlib import Path
from typing import List, Dict, Any, Optional

import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 3. DAPT Configuration

Configure training parameters optimized for domain-adaptive pre-training.

In [ ]:
# =============================================================================
# Model Configuration
# =============================================================================
# Use BASE model for DAPT (not instruct versions)
MODEL_NAME = "Qwen/Qwen3-4B"  # or "Qwen/Qwen3-8B"

# Quantization - OFF by default for best training quality
# Enable only if GPU memory is limited
USE_QUANTIZATION = False
QUANTIZATION_BITS = 4  # 4 or 8

# =============================================================================
# LoRA Configuration - Higher rank for DAPT
# =============================================================================
LORA_R = 128  # Higher rank = more capacity to learn domain knowledge
LORA_ALPHA = 256  # 2x rank is common
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj"
]

# =============================================================================
# Data Configuration
# =============================================================================
DATA_PATH = "./data"  # Path to JSON/JSONL files
TEXT_FIELD = "text"  # Field name containing text
MAX_SEQ_LENGTH = 2048
PACK_SEQUENCES = True  # Pack short documents for efficiency

# =============================================================================
# DAPT Training Configuration
# =============================================================================
OUTPUT_DIR = "./outputs/qwen3_dapt_lora"
NUM_EPOCHS = 3
BATCH_SIZE = 4
GRADIENT_ACCUMULATION = 8  # Larger effective batch

# DAPT uses LOWER learning rate to prevent catastrophic forgetting
LEARNING_RATE = 5e-5  # Lower than fine-tuning (typically 1e-5 to 5e-5)

# Longer warmup for stable DAPT
WARMUP_RATIO = 0.1  # 10% warmup

print("DAPT Configuration:")
print(f"  Model: {MODEL_NAME}")
print(f"  Quantization: {'Enabled' if USE_QUANTIZATION else 'Disabled (full precision)'}")
print(f"  LoRA rank: {LORA_R}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Warmup ratio: {WARMUP_RATIO}")

## 4. Data Loading Utilities

Dataset class with sequence packing for efficient DAPT training.

In [ ]:
def load_json_corpus(data_path: str, text_field: str = "text") -> List[str]:
    """Load text from JSON/JSONL files."""
    data_path = Path(data_path)
    texts = []
    
    def process_item(item):
        if isinstance(item, str):
            return item.strip() if item.strip() else None
        if isinstance(item, dict):
            if text_field in item:
                return str(item[text_field]).strip()
            for field in ["text", "content", "body", "document"]:
                if field in item:
                    return str(item[field]).strip()
            strings = [str(v) for v in item.values() if isinstance(v, str)]
            return " ".join(strings).strip() if strings else None
        return None
    
    def load_file(file_path):
        logger.info(f"Loading {file_path}")
        try:
            if file_path.suffix == ".jsonl":
                with open(file_path, "r", encoding="utf-8") as f:
                    for line in f:
                        if line.strip():
                            try:
                                text = process_item(json.loads(line))
                                if text and len(text) > 10:
                                    texts.append(text)
                            except json.JSONDecodeError:
                                continue
            else:
                with open(file_path, "r", encoding="utf-8") as f:
                    data = json.load(f)
                    items = data if isinstance(data, list) else [data]
                    for item in items:
                        text = process_item(item)
                        if text and len(text) > 10:
                            texts.append(text)
        except Exception as e:
            logger.warning(f"Error loading {file_path}: {e}")
    
    if data_path.is_file():
        load_file(data_path)
    elif data_path.is_dir():
        for pattern in ["**/*.json", "**/*.jsonl"]:
            for fp in sorted(data_path.glob(pattern)):
                load_file(fp)
    
    logger.info(f"Loaded {len(texts)} documents")
    return texts


class DAPTDataset(Dataset):
    """Dataset for DAPT with sequence packing."""
    
    def __init__(self, texts, tokenizer, max_seq_length=2048, pack=True, min_length=64):
        self.tokenizer = tokenizer
        self.max_seq_length = max_seq_length
        self.examples = []
        
        if pack:
            self._pack_texts(texts, min_length)
        else:
            self._tokenize_texts(texts, min_length)
        
        logger.info(f"Created {len(self.examples)} training examples")
    
    def _tokenize_texts(self, texts, min_length):
        for text in texts:
            enc = self.tokenizer(
                text, truncation=True, max_length=self.max_seq_length,
                padding=False, return_tensors=None
            )
            if len(enc["input_ids"]) >= min_length:
                self.examples.append({
                    "input_ids": enc["input_ids"],
                    "attention_mask": enc["attention_mask"]
                })
    
    def _pack_texts(self, texts, min_length):
        """Pack multiple documents into sequences for efficiency."""
        random.shuffle(texts)
        all_tokens = []
        eos_id = self.tokenizer.eos_token_id
        
        for text in texts:
            tokens = self.tokenizer.encode(text, add_special_tokens=False)
            if tokens:
                all_tokens.extend(tokens)
                all_tokens.append(eos_id)  # Document separator
        
        for i in range(0, len(all_tokens), self.max_seq_length):
            chunk = all_tokens[i:i + self.max_seq_length]
            if len(chunk) >= min_length:
                self.examples.append({
                    "input_ids": chunk,
                    "attention_mask": [1] * len(chunk)
                })
    
    def __len__(self):
        return len(self.examples)
    
    def __getitem__(self, idx):
        return self.examples[idx]

## 5. Create Sample Data (Optional)

In [ ]:
# Create sample data for testing (skip if you have your own data)
os.makedirs("./data", exist_ok=True)

sample_data = [
    {"text": "Hypertension, also known as high blood pressure, is a long-term medical condition in which the blood pressure in the arteries is persistently elevated. High blood pressure typically does not cause symptoms. Long-term high blood pressure, however, is a major risk factor for stroke, coronary artery disease, heart failure, atrial fibrillation, peripheral arterial disease, vision loss, chronic kidney disease, and dementia."},
    {"text": "Diabetes mellitus, commonly known as diabetes, is a group of metabolic disorders characterized by a high blood sugar level over a prolonged period of time. Symptoms often include frequent urination, increased thirst and increased appetite. If left untreated, diabetes can cause many health complications. Acute complications can include diabetic ketoacidosis, hyperosmolar hyperglycemic state, or death."},
    {"text": "Cardiovascular disease (CVD) is a class of diseases that involve the heart or blood vessels. CVD includes coronary artery diseases (CAD) such as angina and myocardial infarction (commonly known as a heart attack). Other CVDs include stroke, heart failure, hypertensive heart disease, rheumatic heart disease, cardiomyopathy, abnormal heart rhythms, congenital heart disease, valvular heart disease, carditis, aortic aneurysms, peripheral artery disease, thromboembolic disease, and venous thrombosis."},
    {"text": "The immune system is a complex network of cells, tissues, organs, and the substances they make that helps the body fight infections and other diseases. The immune system includes white blood cells and organs and tissues of the lymph system, such as the thymus, spleen, tonsils, lymph nodes, lymph vessels, and bone marrow."},
    {"text": "Antibiotics, also known as antibacterials, are medications that destroy or slow down the growth of bacteria. They include a range of powerful drugs and are used to treat diseases caused by bacteria. Antibiotics cannot treat viral infections, such as cold, flu, and most coughs."},
]

# Add more comprehensive samples for better DAPT
medical_topics = [
    "pharmacology", "pathology", "anatomy", "physiology", "biochemistry",
    "microbiology", "immunology", "genetics", "epidemiology", "clinical medicine"
]

for i in range(100):
    topic = medical_topics[i % len(medical_topics)]
    sample_data.append({
        "text": f"Domain knowledge in {topic}: This document contains specialized medical terminology, "
                f"clinical information, and scientific concepts related to {topic}. "
                f"Understanding {topic} is essential for medical professionals and researchers. "
                f"Key concepts in {topic} include diagnostic procedures, treatment protocols, "
                f"and evidence-based practice guidelines. Document index: {i}."
    })

with open("./data/sample_corpus.jsonl", "w") as f:
    for item in sample_data:
        f.write(json.dumps(item) + "\n")

print(f"Created {len(sample_data)} sample documents for DAPT")

## 6. Load Model and Tokenizer

Load the base model with optional quantization.

In [ ]:
# Check if using base model (recommended for DAPT)
if "Instruct" in MODEL_NAME or "Chat" in MODEL_NAME:
    print("⚠️  WARNING: Using instruction-tuned model for DAPT.")
    print("   Consider using base model for better knowledge absorption.")

# Create quantization config (optional)
bnb_config = None
if USE_QUANTIZATION:
    if QUANTIZATION_BITS == 4:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=False,
        )
    else:
        bnb_config = BitsAndBytesConfig(load_in_8bit=True)
    print(f"Using {QUANTIZATION_BITS}-bit quantization")
else:
    print("Using full precision (bfloat16) - recommended for DAPT")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    padding_side="right",
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"Tokenizer loaded: {MODEL_NAME}")
print(f"Vocab size: {tokenizer.vocab_size}")

In [ ]:
# Model loading kwargs
model_kwargs = {
    "device_map": "auto",
    "trust_remote_code": True,
}

if bnb_config:
    model_kwargs["quantization_config"] = bnb_config
else:
    model_kwargs["torch_dtype"] = torch.bfloat16

# Try Flash Attention 2
try:
    model_kwargs["attn_implementation"] = "flash_attention_2"
    print("Using Flash Attention 2")
except:
    print("Flash Attention 2 not available")

# Load model
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_kwargs)

# Prepare for training
if bnb_config:
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
else:
    model.gradient_checkpointing_enable()

print(f"Model loaded: {MODEL_NAME}")

## 7. Apply LoRA

Apply LoRA with higher rank for DAPT to learn more domain knowledge.

In [ ]:
# LoRA configuration optimized for DAPT
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# Apply LoRA
model = get_peft_model(model, lora_config)

# Print trainable parameters
model.print_trainable_parameters()

## 8. Prepare Dataset

In [ ]:
# Load corpus
texts = load_json_corpus(DATA_PATH, TEXT_FIELD)
print(f"Loaded {len(texts)} documents")

# Preview first document
if texts:
    print(f"\nFirst document preview:")
    print(f"{texts[0][:300]}...")

In [ ]:
# Create dataset with sequence packing
train_dataset = DAPTDataset(
    texts,
    tokenizer,
    max_seq_length=MAX_SEQ_LENGTH,
    pack=PACK_SEQUENCES,
)

print(f"Dataset size: {len(train_dataset)} packed examples")
print(f"Effective tokens: ~{len(train_dataset) * MAX_SEQ_LENGTH:,}")

## 9. DAPT Training

In [ ]:
# Data collator for causal language modeling
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # Causal LM, not masked LM
)

# Use appropriate optimizer
optim = "paged_adamw_32bit" if USE_QUANTIZATION else "adamw_torch"

# DAPT-optimized training arguments
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    
    # DAPT-specific: Lower learning rate
    learning_rate=LEARNING_RATE,
    
    # DAPT-specific: Longer warmup
    warmup_ratio=WARMUP_RATIO,
    
    # Regularization
    weight_decay=0.01,
    max_grad_norm=1.0,
    
    # Scheduler
    lr_scheduler_type="cosine",
    
    # Logging
    logging_steps=10,
    save_steps=500,
    save_total_limit=3,
    
    # Precision
    fp16=False,
    bf16=True,
    
    # Memory optimization
    gradient_checkpointing=True,
    optim=optim,
    
    seed=42,
    report_to="none",  # Set to "wandb" to use W&B
    remove_unused_columns=False,
    save_safetensors=True,
)

# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
)

print("DAPT Training Configuration:")
print(f"  Effective batch size: {BATCH_SIZE * GRADIENT_ACCUMULATION}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Warmup ratio: {WARMUP_RATIO}")
print(f"  Optimizer: {optim}")

In [ ]:
# Start DAPT training
print("="*60)
print("Starting Domain-Adaptive Pre-Training...")
print("="*60)
trainer.train()

## 10. Save Model

In [ ]:
# Save model and tokenizer
trainer.save_model()
tokenizer.save_pretrained(OUTPUT_DIR)

# Save LoRA adapter separately
lora_output_dir = f"{OUTPUT_DIR}/lora_adapter"
model.save_pretrained(lora_output_dir)

print(f"Model saved to: {OUTPUT_DIR}")
print(f"LoRA adapter saved to: {lora_output_dir}")

## 11. Test Generation

Test the DAPT-trained model with text generation.

In [ ]:
def generate_text(prompt: str, max_new_tokens: int = 150):
    """Generate text with the DAPT-trained base model."""
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512,
    ).to(model.device)
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.pad_token_id,
    )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
# Test prompts related to the domain
test_prompts = [
    "Hypertension is a condition where",
    "The treatment for diabetes includes",
    "Cardiovascular disease can be prevented by",
]

for prompt in test_prompts:
    print(f"\n{'='*60}")
    print(f"Prompt: {prompt}")
    print(f"{'='*60}")
    response = generate_text(prompt)
    print(f"Generated:\n{response}")

## 12. Load Saved Model (Optional)

Load the DAPT-trained model for inference.

In [ ]:
from peft import PeftModel

def load_dapt_model(base_model_name: str, lora_adapter_path: str):
    """Load the DAPT-trained model with LoRA adapter."""
    # Load base model
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=torch.bfloat16,
    )
    
    # Load LoRA adapter
    model = PeftModel.from_pretrained(base_model, lora_adapter_path)
    
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(
        base_model_name,
        trust_remote_code=True,
    )
    
    return model, tokenizer

# Example usage:
# model, tokenizer = load_dapt_model("Qwen/Qwen3-4B", "./outputs/qwen3_dapt_lora/lora_adapter")
print("Load function defined. Uncomment the example to use.")